In [1]:
import pandas as pd
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', 500)
pd.options.display.max_columns = None # show all columns
import numpy as np
import os
import ast
import seaborn as sns

import matplotlib.pyplot as plt
from matplotlib import interactive
interactive(True)

                                                  
pd.set_option('display.max_columns', None)  # show all columns
pd.set_option('display.max_rows', None)

from IPython.display import clear_output
from IPython.display import display, HTML

display(HTML(data="""
<style>
    div#notebook-container    { width: 95%; }
    div#menubar-container     { width: 65%; }
    div#maintoolbar-container { width: 99%; }
</style>
"""))

In [2]:
df = pd.read_csv(r"C:\Users\koand\Downloads\Mintelman_dmin_data.csv")
df2 = pd.read_csv(r"C:\Users\koand\Downloads\Mintelman_dmin_data.csv")

ec_master_ML = pd.read_csv(r"C:\Users\koand\Downloads\ECDNA_Catalogue_MASTER.csv", index_col=0)
ec_full_master = pd.read_csv(r"C:\Users\koand\Downloads\ECDNA_Catalogue_MASTER.csv", index_col=0)

## Build up dataframe of karyotype features - Karyotypes with DMIN

In [ ]:
with_modal_range = []

modal_chr_number = []

XY_chr = []

modal_chr_range = []

marker_chr_num = []



karyotype_modal_dict = {'hexaploid/octaploid':161,'hyperpentaploid':115,'pseudotetraploid':92,'triploid':69,'hypertriploid':75,'hypertetraploid':103,'near-diploid':46,'near-pseudodiploid':46,'polyploid':-1,'hypotetraploid':85,'tetraploid':92,'diploid':46,'hyperdiploid':51, 'hypodiploid':45,'pseudodiploid':46,'near-tetraploid':92,'near-triploid':69,'hypotriploid':66,'iploid':46,'aneuploid':-1, 'hyperpentaploid':121, 'hypopentaploid':110, 'pentaploid':115, 'hypohexaploid':132, 'hexaploid':138, 'heptaploid':161, 'mulitploid':-1, 'heteroploid':-1, 'decaploid':230, 'multiploidy':-1}
def categorize(value):
    if np.isnan(value):
        return np.nan
    closest_category = min(karyotype_modal_dict.keys(), key=lambda k: abs(karyotype_modal_dict[k] - value))
    return closest_category

num = ['1','2','3','4','5','6','7','8','9','10','11','12','13','14','15','16','17','18','19','20','21','22','X','Y']
chrom_loss_dict = dict()
chrom_gain_dict = dict()
for i in num:
    chrom_loss_dict[i]=[]
    chrom_gain_dict[i]=[]


cols = ['INS', 'DEL', 'ADD', 'DUP', 'TRANS', 'INV', 'DER', 'ISO', 'DIC']

chrom_dict = dict()
for i in cols:
    chrom_dict[i]=dict()
    for j in num:
        chrom_dict[i][j]=[]

ins_p = []
add_p = []
del_p=[]
dup_p =[]
inv_p=[]
iso_p=[]
dic_p=[]
trans_p=[]
der_p =[]

for i in df.KaryShort:
    tmp = i.split(',')

    # modal chromosome number
    if '-' in tmp[0]:
        modal_chr_number.append(int(np.mean([int(o) for o in tmp[0].split('-')])))
        with_modal_range.append(tmp[0])
        
        r = [int(o) for o in tmp[0].split('-')]
        modal_chr_range.append(int(np.abs(np.subtract(r[1],r[0]))))
        
        
    else:
        modal_chr_range.append(np.nan)
        if '?' in tmp[0]:
            modal_chr_number.append(np.nan)
        else:
            modal_chr_number.append(int(tmp[0]))
            
    #XY
    if '?' in tmp[1]:
        XY_chr.append(np.nan)
    elif (tmp[1].startswith('X')) or (tmp[1].startswith('Y')):
        XY_chr.append(tmp[1])
    else:
        XY_chr.append(np.nan)
        
    #mar
    mar_counts = [k for k in tmp if ('mar' in k) and (k.startswith('+'))]
    tmp_mar = []
    
    if len(mar_counts)==0:
        marker_chr_num.append(np.nan)
        
    else:
    
        for j in set(mar_counts):
            if j==np.nan:
                tmp_mar.append(np.nan)
            else:
                ll = j.split(',')
                for k in ll:
                    if '+mar' in k:
                        tmp_mar.append(1)
                    else:
                        y = k.replace('+','').split('mar')[0]
                        if '-' in y:
                            tmp_mar.append(int(np.mean([int(o) for o in y.split('-')])))
                        else:
                            tmp_mar.append(int(y))
                            

        if (len(tmp_mar)==1) and (tmp_mar[0]==np.nan):
            marker_chr_num.append(np.nan)
        elif (len(tmp_mar)==1) and (tmp_mar[0]!=np.nan): 
            marker_chr_num.append(tmp_mar[0])
        else:
            marker_chr_num.append(np.max([k for k in tmp_mar if k!=np.nan]))
            
            
            
    #chr loss / gains
    chr_counts = [k for k in tmp if ((k.startswith('-')) or (k.startswith('+'))) and ('mar' not in k) and ('add' not in k) and ('der' not in k) and ('del' not in k) and ('hsr' not in k) and ('+i' not in k) and ('dic' not in k) and ('+t' not in k) and ('+r' not in k) and ('dup' not in k) and ('inv' not in k) and ('r' not in k) and ('?' not in k) and ('dmin' not in k) and ('ma' not in k)]

    
    tmp_gains = []
    tmp_losses = []
    
    for j in chr_counts:
        if j.startswith('-'):
            j = j.split('-')[1].split('/')[0]
            if j != '':
                tmp_losses.append(j)
        elif j.startswith('+'):
            j = j.split('+')[1].split('/')[0]
            if (j != '') and ('-' not in j):
                tmp_gains.append(j)
        else:
            pass
            
    if len(tmp_losses)==0:
        for q in num:
            chrom_loss_dict[q].append(0)
            
    else:
        for l in set(tmp_losses):
            chrom_loss_dict[l].append(1)
            
        for n in [q for q in num if q not in tmp_losses]:
            chrom_loss_dict[n].append(0)
            
    
    if len(tmp_gains)==0:
        for q in num:
            chrom_gain_dict[q].append(0)
            
    else:
        for l in set(tmp_gains):
            chrom_gain_dict[l].append(1)
            
        for n in [q for q in num if q not in tmp_gains]:
            chrom_gain_dict[n].append(0)   
            
    
     # der chrom

    der_counts = [k for k in tmp if ((k.startswith('der')) or (k.startswith('+der'))) and ('mar' not in k) and ('+i' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    
    tmp_der = []
    if (len(der_counts) == 0 ):
        der_p.append(np.nan)
        tmp_der=[]
    else:
        der_p.append(der_counts)
        
        for j in der_counts:
            j = j.replace('?','').split('der(')[1].split(')')[0]
            
            if ';' in j:
                for k in j.split(';'):
                    if k != '':
                        tmp_der.append(k)
            else:
                if j != '':
                    tmp_der.append(j)
             
    if len(tmp_der)==0:
        for q in num:
            chrom_dict['DER'][q].append(0)
            
    elif len(tmp_der)==1:
        chrom_dict['DER'][tmp_der[0]].append(1)

        for n in [q for q in num if q != tmp_der[0]]:
            chrom_dict['DER'][n].append(0)
            
    else:
        for l in set(tmp_der):
            chrom_dict['DER'][l].append(1)
            
        for n in [q for q in num if q not in tmp_der]:
            chrom_dict['DER'][n].append(0)        
            
            
     # dup chrom

    d_counts = [k for k in tmp if ((k.startswith('dup')) or (k.startswith('+dup'))) and ('mar' not in k) and ('+i' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    tmp_counts = []
    if (len(d_counts) == 0 ):
        dup_p.append(np.nan)
        tmp_counts=[]
    else:
        dup_p.append(d_counts)
        
        
        for j in d_counts:
            j = j.replace('?','').split('dup(')[1].split(')')[0]
            
            if ';' in j:
                for k in j.split(';'):
                    if k != '':
                        tmp_counts.append(k)
            else:
                if j != '':
                    tmp_counts.append(j)
    
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['DUP'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['DUP'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['DUP'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['DUP'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['DUP'][n].append(0)                 
            
            
     # del chrom

    d_counts = [k for k in tmp if ((k.startswith('del')) or (k.startswith('+del'))) and ('mar' not in k) and ('+i' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    tmp_counts = []
    if (len(d_counts) == 0 ):
        del_p.append(np.nan)
        tmp_counts=[]
    else:
        del_p.append(d_counts)
        
        
        for j in d_counts:
            j = j.replace('?','').split('del(')[1].split(')')[0]
            
            if ';' in j:
                for k in j.split(';'):
                    if k != '':
                        tmp_counts.append(k)
            else:
                if j != '':
                    tmp_counts.append(j)
    
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['DEL'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['DEL'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['DEL'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['DEL'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['DEL'][n].append(0)                 

     # dic chrom

    d_counts = [k for k in tmp if ((k.startswith('dic')) or (k.startswith('+dic')) or (k.startswith('idic')) or (k.startswith('+idic'))) and ('mar' not in k) and ('+i' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    tmp_counts = []
    if (len(d_counts) == 0 ):
        dic_p.append(np.nan)
        tmp_counts=[]
    else:
        dic_p.append(d_counts)
        
        
        for j in d_counts:
            j = j.replace('?','').split('dic(')[1].split(')')[0]
            
            if ';' in j:
                for k in j.split(';'):
                    if k != '':
                        tmp_counts.append(k)
            else:
                if j != '':
                    tmp_counts.append(j)
    
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['DIC'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['DIC'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['DIC'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['DIC'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['DIC'][n].append(0)           
            
            
            
     # ins chrom

    d_counts = [k for k in tmp if ((k.startswith('ins')) or (k.startswith('+ins'))) and ('mar' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    tmp_counts = []
    if (len(d_counts) == 0 ):
        ins_p.append(np.nan)
        tmp_counts=[]
    else:
        ins_p.append(d_counts)
        
        
        for j in d_counts:
            j = j.replace('?','').split('ins(')[1].split(')')[0]
            
            if ';' in j:
                for k in j.split(';'):
                    if k != '':
                        tmp_counts.append(k)
            else:
                if j != '':
                    tmp_counts.append(j)
     
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['INS'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['INS'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['INS'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['INS'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['INS'][n].append(0)                       
            
            
            
     # inv chrom

    d_counts = [k for k in tmp if ((k.startswith('inv')) or (k.startswith('+inv'))) and ('mar' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    tmp_counts = []
    if (len(d_counts) == 0 ):
        inv_p.append(np.nan)
        tmp_counts=[]
    else:
        inv_p.append(d_counts)
        
        
        for j in d_counts:
            j = j.replace('?','').split('inv(')[1].split(')')[0]
            
            if ';' in j:
                for k in j.split(';'):
                    if k != '':
                        tmp_counts.append(k)
            else:
                if j != '':
                    tmp_counts.append(j)
             
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['INV'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['INV'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['INV'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['INV'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['INV'][n].append(0)                       
            
            
     # iso chrom

    d_counts = [k for k in tmp if ((k.startswith('iso')) or (k.startswith('+iso')) or (k.startswith('+i'))) and ('dic' not in k) and ('mar' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    tmp_counts = []
    if (len(d_counts) == 0 ):
        iso_p.append(np.nan)
        tmp_counts=[]
    else:
        iso_p.append(d_counts)
        
        
        for j in d_counts:
            if 'iso' in j:
                j = j.replace('?','').split('iso(')[1].split(')')[0]
            
                if ';' in j:
                    for k in j.split(';'):
                        if k != '':
                            tmp_counts.append(k)
                else:
                    if j != '':
                        tmp_counts.append(j)
                        
                        
            elif 'i(' in j:
                j = j.replace('?','').split('i(')[1].split(')')[0]
                
                if ';' in j:
                    for k in j.split(';'):
                        if k != '':
                            tmp_counts.append(k)
                else:
                    if j != '':
                        tmp_counts.append(j)      
            else:
                print(j)
    
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['ISO'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['ISO'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['ISO'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['ISO'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['ISO'][n].append(0)           
            
            
     # add chrom

    d_counts = [k for k in tmp if ((k.startswith('add')) or (k.startswith('+add'))) and ('mar' not in k) and ('+i' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    tmp_counts = []
    if (len(d_counts) == 0 ):
        add_p.append(np.nan)
        tmp_counts=[]
    else:
        add_p.append(d_counts)
        
        
        for j in d_counts:
            j = j.replace('?','').split('add(')[1].split(')')[0]
            
            if ';' in j:
                for k in j.split(';'):
                    if k != '':
                        tmp_counts.append(k)
            else:
                if j != '':
                    tmp_counts.append(j)
   
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['ADD'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['ADD'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['ADD'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['ADD'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['ADD'][n].append(0)  
            
              
     # trans chrom

    d_counts = [k for k in tmp if ((k.startswith('trans')) or (k.startswith('+trans')) or (k.startswith('t')) or (k.startswith('+t'))) and ('mar' not in k) and ('+i' not in k) and ('+r' not in k) and ('dmin' not in k)]
    
    tmp_counts = []
    if (len(d_counts) == 0 ):
        trans_p.append(np.nan)
        tmp_counts=[]
    else:
        trans_p.append(d_counts)
        
        
        for j in d_counts:
            if 't(' in j:
                j = j.replace('?','').split('t(')[1].split(')')[0]
            elif 'trans(' in j:
                j = j.replace('?','').split('trans(')[1].split(')')[0]
            elif 'tas(' in j:
                j = j.replace('?','').split('tas(')[1].split(')')[0]
            if ';' in j:
                for k in j.split(';'):
                    if k != '':
                        tmp_counts.append(k)
            else:
                if j != '':
                    tmp_counts.append(j)
             
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['TRANS'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['TRANS'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['TRANS'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['TRANS'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['TRANS'][n].append(0)                       
            
df['ploidy_classification'] = modal_chr_number
df['modal chromosome number'] = modal_chr_number
df['ploidy_classification'] = df['modal chromosome number'].apply(categorize)
df['XY_chromosomes'] = XY_chr
df['marker chromosomes (average #)']=marker_chr_num
df['modal_range_numeric']=modal_chr_range

            
dd = pd.DataFrame(chrom_gain_dict.items()).set_index(0).transpose()


chr_columns_gains = [f'chr_{i}_gains' for i in range(1, 23)]

for i, col in enumerate(chr_columns_gains, start=1):
    df[col] = dd[str(i)].tolist()[0]       
            
            
dd = pd.DataFrame(chrom_loss_dict.items()).set_index(0).transpose()


chr_columns_loss = [f'chr_{i}_loss' for i in range(1, 23)]

for i, col in enumerate(chr_columns_loss, start=1):
    df[col] = dd[str(i)].tolist()[0]
         

# puts in all structural variant data
cols = ['INS', 'DEL', 'ADD', 'DUP', 'TRANS', 'INV', 'DER', 'ISO', 'DIC']
num = ['1','2','3','4','5','6','7','8','9','10','11','12','13','14','15','16','17','18','19','20','21','22']

SV_all = []

for i in cols:

    dd = pd.DataFrame(chrom_dict[i].items()).set_index(0).transpose()
    
    for j in num:
        
        col_name = i+'_'+j

        df[col_name]= dd[j].tolist()[0]
                     
SV_percentages = pd.DataFrame()

for i in num:
    for j in cols:
        l = np.sum(chrom_dict[j][i])
        m = len(chrom_dict[j][i])
        n = np.true_divide(l,m)
        
        SV_percentages = pd.concat([SV_percentages, pd.DataFrame([{'chr':i,'type':j,'number':l,'total_type':m,'percentage_type':n}])])
    

tmpdf = df[df.columns.tolist()[58:]]

for i in df.index:
    
    SV_all.append(np.sum(tmpdf[tmpdf.index==i].transpose()).values[0])

df['SV_sum_all_events']=SV_all


df.head(25)
            
        

In [4]:
ec_master_ML.primary_disease.unique().tolist()

['Lung Cancer',
 'Leukemia',
 'Ovarian Cancer',
 'Prostate Cancer',
 'Colon/Colorectal Cancer',
 'Brain Cancer',
 'Gastric Cancer',
 'Breast Cancer',
 'Bone Cancer',
 'Bladder Cancer',
 'Neuroblastoma',
 'Eye Cancer',
 'Sarcoma',
 'Rhabdoid',
 'Cervical Cancer',
 'Thyroid Cancer',
 'Skin Cancer',
 'Kidney Cancer',
 nan,
 'Head and Neck Cancer',
 'Pancreatic Cancer',
 'Myeloma',
 'Lymphoma',
 'Endometrial/Uterine Cancer',
 'Esophageal Cancer',
 'Bile Duct Cancer',
 'Fibroblast',
 'Liver Cancer',
 'Adrenal Cancer',
 'Non-Cancerous',
 'Embryonal Cancer',
 'Liposarcoma',
 'Engineered',
 'Gallbladder Cancer',
 'Unknown',
 'Teratoma']

In [6]:
df.Morph.unique().tolist() 

['Acute myeloblastic leukemia with maturation (FAB type M2)',
 'Acute lymphoblastic leukemia/lymphoblastic lymphoma',
 'Myelodysplastic/myeloproliferative disease, NOS',
 'Plasma cell leukemia',
 'Adenocarcinoma',
 'Acute myelomonocytic leukemia (FAB type M4)',
 'Acute erythroleukemia (FAB type M6)',
 'Neuroblastoma',
 'Chronic myeloid leukemia, t(9;22)',
 'Refractory anemia with excess blasts-2',
 'Leiomyosarcoma',
 'Mastocytosis',
 'Undifferentiated pleomorphic sarcoma',
 'Carcinoma, NOS',
 'Peripheral neuroepithelioma',
 'Clear cell sarcoma',
 'Acute myeloid leukemia, NOS',
 'Squamous cell carcinoma',
 'Primitive neuroectodermal tumor/Medulloblastoma',
 'Undifferentiated carcinoma, small cell',
 'Carcinoid tumor',
 'Astrocytoma, grade III-IV/Glioblastoma',
 'Alveolar rhabdomyosarcoma',
 'Acute promyelocytic leukemia (FAB type M3)',
 'Acute myeloblastic leukemia without maturation (FAB type M1)',
 'T-prolymphocytic leukemia',
 'Osteosarcoma, NOS',
 'Wilms tumor',
 'Acute myeloblastic

In [10]:
morphs = ['lung', 'leukemia', 'ovarian', 'prostate', 'colon', 'brain', 'gastric', 'breast', 'bone', 'bladder',
            'neuroblastoma', 'eye', 'sarcoma', 'rhabdoid', 'cervical', 'thyroid', 'skin', 'kidney', 'head and neck',
            'pancreatic', 'myeloma', 'lymphoma', 'endometrial/uterine', 'esophageal', 'bile duct', 'fibroblast',
            'liver', 'adrenal', 'non-cancerous', 'embryonal', 'liposarcoma', 'egineered', 'gallbladder', 'unknown', 'teratoma']

morph_dict = {}

for i in df.index:
    morph_value = df.loc[i, 'Morph']
    if any(keyword in morph_value for morph in morphs):
        morph_dict[i] = morph_value

morph_dict

{0: 'Acute myeloblastic leukemia with maturation (FAB type M2)',
 1: 'Acute lymphoblastic leukemia/lymphoblastic lymphoma',
 3: 'Plasma cell leukemia',
 4: 'Acute myeloblastic leukemia with maturation (FAB type M2)',
 5: 'Acute myeloblastic leukemia with maturation (FAB type M2)',
 7: 'Acute myelomonocytic leukemia (FAB type M4)',
 8: 'Acute erythroleukemia (FAB type M6)',
 9: 'Acute myeloblastic leukemia with maturation (FAB type M2)',
 11: 'Acute myeloblastic leukemia with maturation (FAB type M2)',
 13: 'Acute myelomonocytic leukemia (FAB type M4)',
 14: 'Chronic myeloid leukemia, t(9;22)',
 15: 'Acute myeloblastic leukemia with maturation (FAB type M2)',
 18: 'Leiomyosarcoma',
 21: 'Chronic myeloid leukemia, t(9;22)',
 23: 'Undifferentiated pleomorphic sarcoma',
 26: 'Clear cell sarcoma',
 27: 'Acute myeloid leukemia, NOS',
 35: 'Alveolar rhabdomyosarcoma',
 36: 'Acute promyelocytic leukemia (FAB type M3)',
 38: 'Acute myeloblastic leukemia with maturation (FAB type M2)',
 40: 'Acu

In [5]:
cells_to_throw = []
cells_withX = []
cells_with_parsed_data=[]
parsed_xy = []
manual_xy_dict = {}

manual_xy_dict['MP46_UVEA'] = np.nan
manual_xy_dict['A375_SKIN_CJ1_RESISTANT'] = np.nan
manual_xy_dict['U2904_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE'] = np.nan
manual_xy_dict['L82_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE'] = np.nan
manual_xy_dict['MCCAR_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE'] = np.nan
manual_xy_dict['HAL01_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE'] = np.nan
manual_xy_dict['BALL1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE'] = np.nan
manual_xy_dict['NCIH835_LUNG'] = np.nan
manual_xy_dict['NCIH748_LUNG'] = np.nan
manual_xy_dict['NCIH720_LUNG'] = np.nan
manual_xy_dict['NCIH250_LUNG'] = np.nan
manual_xy_dict['NCIH64_LUNG'] = np.nan
manual_xy_dict['FLO1_OESOPHAGUS'] = np.nan
manual_xy_dict['FARAGE_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE'] = np.nan
manual_xy_dict['A388_SKIN'] = np.nan
manual_xy_dict['SW13_ADRENAL_CORTEX'] = np.nan
manual_xy_dict['ME180_CERVIX'] = np.nan
manual_xy_dict['MDAMB330_BREAST'] = np.nan
manual_xy_dict['DOTC24510_CERVIX'] = np.nan
manual_xy_dict['TT_THYROID'] = np.nan
manual_xy_dict['SKNEP1_BONE'] = np.nan
manual_xy_dict['NCIH2141_LUNG'] = np.nan
manual_xy_dict['M059J_CENTRAL_NERVOUS_SYSTEM'] = 'Y'
manual_xy_dict['HS604T_FIBROBLAST'] = np.nan
manual_xy_dict['LNCAPCLONEFGC_PROSTATE'] = np.nan
manual_xy_dict['HEC1A_ENDOMETRIUM'] = np.nan
manual_xy_dict['NCIH650_LUNG'] = np.nan
manual_xy_dict['HEC1B_ENDOMETRIUM'] = np.nan
manual_xy_dict['SKUT1_SOFT_TISSUE'] = np.nan
manual_xy_dict['MDAMB361_BREAST'] = np.nan
manual_xy_dict['MDAMB436_BREAST'] = np.nan
manual_xy_dict['HS819T_FIBROBLAST'] = np.nan
manual_xy_dict['PFEIFFER_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE'] = np.nan
manual_xy_dict['HS294T_SKIN'] = 'Y'
manual_xy_dict['KYSE450_OESOPHAGUS'] = np.nan
manual_xy_dict['HS698T_FIBROBLAST'] = np.nan
manual_xy_dict['FADU_UPPER_AERODIGESTIVE_TRACT'] = np.nan
manual_xy_dict['NCIH1373_LUNG'] = np.nan
manual_xy_dict['NCIH1373_LUNG'] = np.nan
manual_xy_dict['BT483_BREAST'] = np.nan
manual_xy_dict['HS695T_SKIN'] = 'Y'
manual_xy_dict['HLFA_FIBROBLAST'] = np.nan
manual_xy_dict['KIJK_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE'] = np.nan
manual_xy_dict['HCC2218_BREAST'] = np.nan
manual_xy_dict['DM3_FIBROBLAST'] = np.nan
manual_xy_dict['HCC2157_BREAST'] = np.nan
manual_xy_dict['HCC70_BREAST'] = np.nan
manual_xy_dict['NCIH1355_LUNG'] = np.nan
manual_xy_dict['NCIH28_PLEURA'] = np.nan
manual_xy_dict['HCC1806_BREAST'] = np.nan
manual_xy_dict['MIAPACA2_PANCREAS'] = np.nan
manual_xy_dict['CALU1_LUNG'] = '-Y'
manual_xy_dict['SNU423_LIVER'] = np.nan
manual_xy_dict['SNU182_LIVER'] = np.nan
manual_xy_dict['SNU387_LIVER'] = np.nan
manual_xy_dict['SH4_SKIN'] = np.nan
manual_xy_dict['SNU475_LIVER'] = np.nan
manual_xy_dict['SNU449_LIVER'] = np.nan
manual_xy_dict['HS739T_FIBROBLAST'] = np.nan
manual_xy_dict['NCIH647_LUNG'] = np.nan
manual_xy_dict['U2OS_BONE'] = np.nan
manual_xy_dict['MG63_BONE'] = np.nan
manual_xy_dict['HCC1428_BREAST'] = np.nan
manual_xy_dict['HCC1500_BREAST'] = np.nan
manual_xy_dict['EFO21_OVARY'] = 'None'
manual_xy_dict['HS688AT_FIBROBLAST'] = np.nan
manual_xy_dict['HS852T_SKIN'] = np.nan
manual_xy_dict['HPAC_PANCREAS'] = np.nan
manual_xy_dict['CALU6_LUNG'] = '-Y'
manual_xy_dict['COLO201_LARGE_INTESTINE'] = np.nan
manual_xy_dict['HS600T_FIBROBLAST'] = np.nan
manual_xy_dict['HS742T_FIBROBLAST'] = np.nan
manual_xy_dict['ASPC1_PANCREAS'] = np.nan
manual_xy_dict['SNU398_LIVER'] = np.nan
manual_xy_dict['A375_SKIN'] = np.nan
manual_xy_dict['HS675T_FIBROBLAST'] = np.nan
manual_xy_dict['HS255T_FIBROBLAST'] = np.nan
manual_xy_dict['BHT101_THYROID'] = np.nan
manual_xy_dict['HS737T_FIBROBLAST'] = np.nan
manual_xy_dict['HS343T_FIBROBLAST'] = np.nan
manual_xy_dict['HS839T_FIBROBLAST'] = np.nan
manual_xy_dict['HS766T_PANCREAS'] = np.nan
manual_xy_dict['RD_SOFT_TISSUE'] = np.nan
manual_xy_dict['PANC1_PANCREAS'] = np.nan
manual_xy_dict['SW1990_PANCREAS'] = np.nan
manual_xy_dict['M059K_CENTRAL_NERVOUS_SYSTEM'] = np.nan
manual_xy_dict['CFPAC1_PANCREAS'] = np.nan
manual_xy_dict['HS940T_FIBROBLAST'] = np.nan
manual_xy_dict['HS274T_FIBROBLAST'] = np.nan
manual_xy_dict['HCC1187_BREAST'] = np.nan
manual_xy_dict['HS616T_FIBROBLAST'] = np.nan
manual_xy_dict['MDAMB134VI_BREAST'] = np.nan
manual_xy_dict['A101D_SKIN'] = np.nan
manual_xy_dict['CACO2_LARGE_INTESTINE'] = np.nan
manual_xy_dict['SCABER_URINARY_TRACT'] = 'Y'
manual_xy_dict['SW1116_LARGE_INTESTINE'] = np.nan
manual_xy_dict['KATOIII_STOMACH'] = np.nan
manual_xy_dict['ZR7530_BREAST'] = np.nan
manual_xy_dict['UACC812_BREAST'] = np.nan
manual_xy_dict['BT474_BREAST'] = np.nan
manual_xy_dict['SW620_LARGE_INTESTINE'] = np.nan
manual_xy_dict['NCIH1930_LUNG'] = np.nan
manual_xy_dict['NCIH1048_LUNG'] = np.nan
manual_xy_dict['NCIH1436_LUNG'] = np.nan
manual_xy_dict['DMS79_LUNG'] = np.nan
manual_xy_dict['NCIH841_LUNG'] = np.nan
manual_xy_dict['NCIH211_LUNG'] = np.nan
manual_xy_dict['786O_KIDNEY'] = 'Y'
manual_xy_dict['NCIH226_LUNG'] = np.nan
manual_xy_dict['SKMEL2_SKIN'] = np.nan
manual_xy_dict['COLO205_LARGE_INTESTINE'] = np.nan
manual_xy_dict['HCC1569_BREAST'] = np.nan
manual_xy_dict['UACC893_BREAST'] = np.nan
manual_xy_dict['HCC1419_BREAST'] = np.nan
manual_xy_dict['HT144_SKIN_FV1_RESISTANT'] = np.nan
manual_xy_dict['T47D_BREAST'] = np.nan
manual_xy_dict['HT3_CERVIX'] = np.nan
manual_xy_dict['SKCO1_LARGE_INTESTINE'] = np.nan
manual_xy_dict['HT144_SKIN'] = 'None'
manual_xy_dict['DETROIT562_UPPER_AERODIGESTIVE_TRACT'] = np.nan
manual_xy_dict['CAOV4_OVARY'] = np.nan
manual_xy_dict['SKPNDW_BONE'] = np.nan
manual_xy_dict['MCIXC_AUTONOMIC_GANGLIA'] = np.nan
manual_xy_dict['CAKI1_KIDNEY'] = np.nan
manual_xy_dict['HCC1395_BREAST'] = np.nan
manual_xy_dict['NCIH508_LARGE_INTESTINE'] = np.nan
manual_xy_dict['NCIN87_STOMACH'] = np.nan
manual_xy_dict['NIHOVCAR3_OVARY'] = np.nan
manual_xy_dict['NCIH1092_LUNG'] = np.nan
manual_xy_dict['NCIH1694_LUNG'] = np.nan
manual_xy_dict['NCIH69_LUNG'] = np.nan
manual_xy_dict['184B5_BREAST'] = 'XX'
manual_xy_dict['184A1_BREAST'] = 'XX'
manual_xy_dict['94T778_SOFT_TISSUE'] = 'XX'
manual_xy_dict['93T449_SOFT_TISSUE'] = 'XX'
manual_xy_dict['WPE1NA22_PROSTATE'] = 'X'
manual_xy_dict['SW954_CERVIX'] = 'XX'
manual_xy_dict['BJHTERT_FIBROBLAST'] = 'XY'
manual_xy_dict['OV90_OVARY'] = 'XX'
manual_xy_dict['SW954_VULVA'] = 'XX'
manual_xy_dict['NCIH820_LUNG'] = 'XYY/XXYY'
manual_xy_dict['BJHTERT_SKIN'] = 'XY'
manual_xy_dict['SW872_SOFT_TISSUE'] = 'X'
manual_xy_dict['NTERA2CLD1_TESTIS'] = 'Y'
manual_xy_dict['H9_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE'] = 'None'
manual_xy_dict['PFSK1_CENTRAL_NERVOUS_SYSTEM'] = 'XXY'
manual_xy_dict['MS751_CERVIX'] = 'None'
manual_xy_dict['NCIH292_LUNG'] = 'XX'
manual_xy_dict['LS411N_LARGE_INTESTINE'] = 'X'
manual_xy_dict['LS180_LARGE_INTESTINE'] = np.nan
manual_xy_dict['ES2_OVARY'] = 'XX'
manual_xy_dict['MDAMB415_BREAST'] = 'XXXX/XXXXX'
manual_xy_dict['CHAGOK1_LUNG'] = 'None'
manual_xy_dict['MDAMB468_BREAST'] = 'X'
manual_xy_dict['NCIH2126_LUNG'] = 'XX'
manual_xy_dict['MDAMB175VII_BREAST'] = 'XX/XXX/XXXX'
manual_xy_dict['SJSA1_BONE'] = 'XY'
manual_xy_dict['SW900_LUNG'] = 'X'
manual_xy_dict['NCIH441_LUNG'] = 'XY'
manual_xy_dict['MDAMB157_BREAST'] = 'XXX'
manual_xy_dict['UMUC3_URINARY_TRACT'] = 'X'
manual_xy_dict['SW1353_BONE'] = 'XX'
manual_xy_dict['J82_URINARY_TRACT'] = 'XY'
manual_xy_dict['NCIH520_LUNG'] = 'X'
manual_xy_dict['G402_SOFT_TISSUE'] = 'XX'
manual_xy_dict['SKLU1_LUNG'] = 'XX'
manual_xy_dict['LS1034_LARGE_INTESTINE'] = 'YY'
manual_xy_dict['GA10_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE'] = 'XY'
manual_xy_dict['SKNSH_AUTONOMIC_GANGLIA'] = 'XX'
manual_xy_dict['SKLMS1_SOFT_TISSUE'] = 'XX'
manual_xy_dict['G401_SOFT_TISSUE'] = 'XY'
manual_xy_dict['LS513_LARGE_INTESTINE'] = 'XY'
manual_xy_dict['C33A_CERVIX'] = 'XX'
manual_xy_dict['SKMEL5_SKIN'] = 'XXX'
manual_xy_dict['BT549_BREAST'] = 'XXX/XXXX'
manual_xy_dict['NCIH209_LUNG'] = 'XY'
manual_xy_dict['SKNDZ_AUTONOMIC_GANGLIA'] = 'XX'
manual_xy_dict['CALU3_LUNG'] = 'XX'
manual_xy_dict['SKOV3_OVARY'] = 'X/XX'
manual_xy_dict['NCIH522_LUNG'] = 'XXY'
manual_xy_dict['SKBR3_BREAST'] = 'XXX'
manual_xy_dict['SNU1_STOMACH'] = 'XY'
manual_xy_dict['AGS_STOMACH'] = 'X'
manual_xy_dict['T98G_CENTRAL_NERVOUS_SYSTEM'] = 'X'
manual_xy_dict['MESSA_SOFT_TISSUE'] = 'XX'
manual_xy_dict['SW684_SOFT_TISSUE'] = 'XXY'
manual_xy_dict['SW982_SOFT_TISSUE'] = 'XX'
manual_xy_dict['HS746T_STOMACH'] = 'XY'
manual_xy_dict['SW1088_CENTRAL_NERVOUS_SYSTEM'] = 'XXYY'
manual_xy_dict['769P_KIDNEY'] = 'XX'
manual_xy_dict['D341MED_CENTRAL_NERVOUS_SYSTEM'] = 'XY'
manual_xy_dict['CCFSTTG1_CENTRAL_NERVOUS_SYSTEM'] = 'XX'
manual_xy_dict['T84_LARGE_INTESTINE'] = 'None'
manual_xy_dict['MDAMB435S_SKIN'] = 'XX'
manual_xy_dict['NCIH747_LARGE_INTESTINE'] = 'XY'
manual_xy_dict['NCIH716_LARGE_INTESTINE'] = 'X'
manual_xy_dict['SW626_LARGE_INTESTINE'] = 'XXXX'
manual_xy_dict['NCIH82_LUNG'] = 'XX'
manual_xy_dict['NCIH446_LUNG']= 'XXYY'
manual_xy_dict['NCIH510_LUNG'] = 'XX'
manual_xy_dict['SNU16_STOMACH'] = 'XXX'
manual_xy_dict['SNUC2A_LARGE_INTESTINE'] = 'XX'
manual_xy_dict['HCC1937_BREAST'] = 'der(X)'
manual_xy_dict['DAOY_CENTRAL_NERVOUS_SYSTEM'] = 'XX'
manual_xy_dict['U87MG_CENTRAL_NERVOUS_SYSTEM'] = 'X'
manual_xy_dict['SNU5_STOMACH'] = 'X'
manual_xy_dict['SW1783_CENTRAL_NERVOUS_SYSTEM'] = 'XXXYY'
manual_xy_dict['HS683_CENTRAL_NERVOUS_SYSTEM'] = 'XXYY'
manual_xy_dict['HCC38_BREAST'] = 'None'
manual_xy_dict['SIHA_CERVIX'] = 'XX'
manual_xy_dict['NCIH596_LUNG'] = 'XX'
manual_xy_dict['CAMA1_BREAST'] = 'XX'
manual_xy_dict['NCIH460_LUNG'] = 'XXYY'
manual_xy_dict['SKMEL28_SKIN'] = 'XXY'
manual_xy_dict['NCIH146_LUNG'] = 'XX'
manual_xy_dict['D283MED_CENTRAL_NERVOUS_SYSTEM'] = 'X'
manual_xy_dict['ZR751_BREAST'] = 'XXX/XXXX'
manual_xy_dict['H4_CENTRAL_NERVOUS_SYSTEM'] = 'XXYY'
manual_xy_dict['MALME3M_SKIN'] = 'XXYY'
manual_xy_dict['LS123_LARGE_INTESTINE'] = 'XXX'
manual_xy_dict['SKMEL31_SKIN'] = 'XX'
manual_xy_dict['NCIH661_LUNG'] = 'XXYY'
manual_xy_dict['EB2_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE'] = 'XXXX'
manual_xy_dict['SNUC2B_LARGE_INTESTINE'] = 'XX'
manual_xy_dict['NCIH345_LUNG'] = 'XX'
manual_xy_dict['SW962_VULVA'] = 'XXX'
manual_xy_dict['SW626_OVARY'] = 'XXXX'

ec_master = ec_master_ML

ec_master = ec_master[ec_master['Available karyotype']=='Y']


for i in ec_master.index:
    
    if i in manual_xy_dict.keys():
        parsed_xy.append(manual_xy_dict[i])
        cells_with_parsed_data.append(i)
    else:
    
        t1 = ec_master.loc[i,'DSMZ_karyotype']
        t2 = ec_master.loc[i,'ATCC_karyotype_text']

        if (pd.notna(t1)) and ('X' in t1):
            cells_withX.append(i)

        if (pd.notna(t2)) and ('X' in t2):
            cells_withX.append(i)

        if pd.notna(t1):
            if ('>X' in t1) or ('> X') in t1 or ('>,X' in t1) or ('>der(' in t1):

                for j in t1.split(','):
                    if ('>X' in j) or ('> X' in j) or ('>,X' in j) or ('>der(' in j):
                        j = j.split('>')[1]
                        if ';' in j:
                            parsed_xy.append(j.split(';')[0])
                            cells_with_parsed_data.append(i)
                        elif 'rearrangements' in j:
                            parsed_xy.append(j.split(' -')[0])
                            cells_with_parsed_data.append(i)
                        else:
                            parsed_xy.append(j)
                            cells_with_parsed_data.append(i)
                            
                    if ('X' in j) and ('>,X' in t1):
                        if ';' in j:
                            parsed_xy.append(j.split(';')[0])
                            cells_with_parsed_data.append(i)
                        elif 'rearrangements' in j:
                            parsed_xy.append(j.split(' -')[0])
                            cells_with_parsed_data.append(i)
                        else:
                            parsed_xy.append(j)
                            cells_with_parsed_data.append(i)
            else:

                if pd.notna(t2):
                    if ('>X' in t2) or ('> X' in t2) or ('>,X' in t2) or ('>der(' in t2):
                        for j in t2.split(','):
                            if ('>X' in t2) or ('> X' in t2) or ('>,X' in t2) or ('>der(' in t2):

                                j = j.split('>')[1]
                                if 'rearrangements' in j:
                                    cells_with_parsed_data.append(i)
                                    parsed_xy.append(j.split(' -')[0])
                                else:
                                    cells_with_parsed_data.append(i)
                                    parsed_xy.append(j)
                                    

                    else:
                        cells_to_throw.append(i)
                        parsed_xy.append(np.nan)
                        cells_with_parsed_data.append(i)
                
                    
print(len(cells_with_parsed_data), len(ec_master))

ec_master['XY_chromosomes'] = parsed_xy

ec_master.head()

600 600


C:\Users\koand\AppData\Local\Temp\ipykernel_30552\3770341009.py:306: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ec_master['XY_chromosomes'] = parsed_xy


,ECDNA,HSR,Validation source,WGS coordinates,ATAC coordinates,ATAC representative,Long read DNA,RNA Coordinates,DepMap_CNV,DepMap_GE,DepMap_CRISPR,DepMap (CNV),AA prediction,AA -AMP Type,AA - genes_on_ecDNA,CH prediction,CH - genes on ecDNA,cellosaurus,ATCC,DSMZ,Available karyotype,K_Parsing_complete,ATCC_karyotype_text,ATCC_DM,DSMZ_karyotype,DSMZ_DM,DepMap_ID,cell_line_name,stripped_cell_line_name,alias,COSMICID,AA_Oncogenes,AA_All_genes,AA_Complexity_scores,XY_chromosomes
CCLE_Name,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
HCC827_LUNG,Y,NaN,Turner et al. 2017 Nature,SRR8639147,"SRR17490254,SRR17490255,SRR17490256,SRR1749025...",SRR17490254,NaN,"SRR25516004,SRR25516005,SRR25516006,SRR2551600...",NaN,NaN,NaN,NaN,Y,"HSR, DMs",NaN,NaN,NaN,HCC827,CRL-2868,ACC-566,Y,Y,NaN,NaN,human flat-moded hypotriploid karyotype with 6...,Y,ACH-000012,HCC827,HCC827,NaN,1240146.0,"[""['EGFR']"", '[]', ""['MYC', 'PVT1']""]","[""['EGFR', 'EGFR-AS1', 'SEC61G', 'VSTM2A', 'VS...","[0.3771627530312313, 0.5844876817892763, 0.953...",XXX/XXXX
HL60_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,Y,NaN,"Turner et al. 2017 Nature, ATCC",SRR4009290,"SRR22102206,SRR22102207,SRR22102208,SRR1339447...",SRR22102206,NaN,"ERR10492830,ERR10492829,ERR10492828,ERR1049282...",NaN,NaN,NaN,NaN,Y,Circular,NaN,Y,NaN,HL-60,CCL-240,ACC-3,Y,Y,The stemline chromosome number is pseudodiploi...,Y,human flat-moded hypotetraploid karyotype with...,Y,ACH-000002,HL-60,HL60,NaN,905938.0,NaN,NaN,NaN,XX
NCIH82_LUNG,Y,NaN,Pongor et al. Cancer Discovery 2023; https://w...,SRR8670699,"SRR12461581,SRR12461580,SRR12461579,SRR1246157...",SRR12461581,NaN,"SRR21986728,SRR21986729,SRR21986730,SRR2198673...",NaN,NaN,NaN,NaN,Y,Complex,c-MYC,Y,NaN,NCI-H82,HTB-175,ACC-556,Y,Y,This is a near triploid human cell line. The m...,Y,NaN,NaN,ACH-000355,NCI-H82,NCIH82,NaN,688031.0,"[""['MYC']"", '[]']","[""['CASC11', 'CASC19', 'CASC21', 'CASC8', 'CCA...","[0.9682448034987724, 0.48222712722266]",XX
NCIH69_LUNG,Y,NaN,Pongor et al. Cancer Discovery 2023,SRR8670701,"SRR7213141,SRR7213140,SRR7213139",SRR7213141,NaN,"SRR20827404,SRR20827405,SRR20827406,SRR1738206...",NaN,NaN,NaN,MYCN(4.85),Y,Complex,MYC-N,Y,NaN,NCI-H69,HTB-119,NaN,Y,Y,modal number = 76 to 78; range = 40 to 87This ...,N,NaN,NaN,ACH-000358,NCI-H69,NCIH69,NaN,688027.0,"[""['MYCN']"", '[]']","[""['LRRTM4-AS1', 'MYCN', 'MYCNOS', 'MYCNUT']"",...","[1.0604994273391384, 0.6924720945389742]",NaN
NCIH1694_LUNG,Y,NaN,Pongor et al. Cancer Discovery 2023,SRR8652045,NaN,NaN,NaN,"SRR11581914,SRR11581913,SRR11581912,SRR1158191...",NaN,NaN,NaN,MYCL(4.8),Y,Complex,MYC-L,NaN,NaN,NCI-H1694,CRL-5888,NaN,Y,Y,del(p21-pter),N,NaN,NaN,ACH-000431,NCI-H1694,NCIH1694,NaN,688006.0,"['[]', ""['ARHGEF2']""]","[""['AKT3', 'BMP8A', 'BMP8B', 'C1orf198', 'CD84...","[1.425553528959825, 0.5773941705943326]",NaN


In [6]:
cells_for_further_tests = [i for i in ec_master.index if i not in parsed_xy]

for i in cells_for_further_tests:
    t1 = ec_master.loc[i,'DSMZ_karyotype']
    t2 = ec_master.loc[i,'ATCC_karyotype_text']
    
    if (pd.notna(t1)) and ('X' in t1):
        for j in t1.split(','):
            if ('X' in j) and ('>' not in j):
                pass
        
    if (pd.notna(t2)) and ('X' in t2):
        for j in t2.split(','):
            if ('X' in j):
                if 'copies' in j:
                    pass
            else:
                pass
for i in [i for i in ec_master.index if i not in cells_with_parsed_data]:
    t1 = ec_master.loc[i,'DSMZ_karyotype']
    t2 = ec_master.loc[i,'ATCC_karyotype_text']

NCIH1836_LUNG
nan
del(p11-qter)

HCC202_BREAST
nan
polyploid



In [8]:
remove = []

ec_samples = ec_master.drop(columns={'Available karyotype','ATCC_karyotype_text','DSMZ_karyotype'},axis=1)
ec_samples['dataset'] = ['DepMap' for i in ec_samples.index]
ec_samples = ec_samples.reset_index()
ec_samples = ec_samples.rename(columns={'Y/N/P':'ECDNA_classification','CCLE_Name':'Sample_ID'})

ec_samples.head()

for i in ec_samples.index:
    cell = ec_samples.loc[i,'Sample_ID']
    if cell in ec_full_master.index:
        if pd.notna(ec_full_master.loc[cell,'ECDNA']):
            pass
        else:
            remove.append(i)

print(len(ec_samples))
ec_samples = ec_samples.drop(remove)
print(len(ec_samples))


600
554


In [9]:
ecDNA_list = []
HSR_list = []


for i in ec_samples.index:
    cell = ec_samples.loc[i,'Sample_ID']
    if cell in ec_full_master.index:
        ecDNA_list.append(ec_full_master.loc[cell,'ECDNA'])
        HSR_list.append(ec_full_master.loc[cell,'HSR'])

    else:
        print(cell)
    
print((len(ec_samples)),len(ecDNA_list),len(HSR_list))

554 554 554


In [10]:
ec_samples['ECDNA_classification'] = ecDNA_list
ec_samples['HSR_classification'] = HSR_list

df['ECDNA_classification'] = ['Y' for i in df.index]
df['dataset'] = ['MITELMAN' for i in df.index]
df = df.drop(columns={'CaseNo','InvNo','Abbreviation','Journal','Topo','KaryShort','KaryLong'})

df = df.rename(columns={'Refno':'Sample_ID'})

HSR_lst2 = []

for i in df2.KaryShort:
    if 'hsr' in i.lower():
        HSR_lst2.append('Y')
    else:
        HSR_lst2.append('N')

df['HSR_classification'] = HSR_lst2
df.head()

,Sample_ID,ploidy_classification,modal chromosome number,XY_chromosomes,marker chromosomes (average #),modal_range_numeric,chr_1_gains,chr_2_gains,chr_3_gains,chr_4_gains,chr_5_gains,chr_6_gains,chr_7_gains,chr_8_gains,chr_9_gains,chr_10_gains,chr_11_gains,chr_12_gains,chr_13_gains,chr_14_gains,chr_15_gains,chr_16_gains,chr_17_gains,chr_18_gains,chr_19_gains,chr_20_gains,chr_21_gains,chr_22_gains,chr_1_loss,chr_2_loss,chr_3_loss,chr_4_loss,chr_5_loss,chr_6_loss,chr_7_loss,chr_8_loss,chr_9_loss,chr_10_loss,chr_11_loss,chr_12_loss,chr_13_loss,chr_14_loss,chr_15_loss,chr_16_loss,chr_17_loss,chr_18_loss,chr_19_loss,chr_20_loss,chr_21_loss,chr_22_loss,INS_1,INS_2,INS_3,INS_4,INS_5,INS_6,INS_7,INS_8,INS_9,INS_10,INS_11,INS_12,INS_13,INS_14,INS_15,INS_16,INS_17,INS_18,INS_19,INS_20,INS_21,INS_22,DEL_1,DEL_2,DEL_3,DEL_4,DEL_5,DEL_6,DEL_7,DEL_8,DEL_9,DEL_10,DEL_11,DEL_12,DEL_13,DEL_14,DEL_15,DEL_16,DEL_17,DEL_18,DEL_19,DEL_20,DEL_21,DEL_22,ADD_1,ADD_2,ADD_3,ADD_4,ADD_5,ADD_6,ADD_7,ADD_8,ADD_9,ADD_10,ADD_11,ADD_12,ADD_13,ADD_14,ADD_15,ADD_16,ADD_17,ADD_18,ADD_19,ADD_20,ADD_21,ADD_22,DUP_1,DUP_2,DUP_3,DUP_4,DUP_5,DUP_6,DUP_7,DUP_8,DUP_9,DUP_10,DUP_11,DUP_12,DUP_13,DUP_14,DUP_15,DUP_16,DUP_17,DUP_18,DUP_19,DUP_20,DUP_21,DUP_22,TRANS_1,TRANS_2,TRANS_3,TRANS_4,TRANS_5,TRANS_6,TRANS_7,TRANS_8,TRANS_9,TRANS_10,TRANS_11,TRANS_12,TRANS_13,TRANS_14,TRANS_15,TRANS_16,TRANS_17,TRANS_18,TRANS_19,TRANS_20,TRANS_21,TRANS_22,INV_1,INV_2,INV_3,INV_4,INV_5,INV_6,INV_7,INV_8,INV_9,INV_10,INV_11,INV_12,INV_13,INV_14,INV_15,INV_16,INV_17,INV_18,INV_19,INV_20,INV_21,INV_22,DER_1,DER_2,DER_3,DER_4,DER_5,DER_6,DER_7,DER_8,DER_9,DER_10,DER_11,DER_12,DER_13,DER_14,DER_15,DER_16,DER_17,DER_18,DER_19,DER_20,DER_21,DER_22,ISO_1,ISO_2,ISO_3,ISO_4,ISO_5,ISO_6,ISO_7,ISO_8,ISO_9,ISO_10,ISO_11,ISO_12,ISO_13,ISO_14,ISO_15,ISO_16,ISO_17,ISO_18,ISO_19,ISO_20,ISO_21,ISO_22,DIC_1,DIC_2,DIC_3,DIC_4,DIC_5,DIC_6,DIC_7,DIC_8,DIC_9,DIC_10,DIC_11,DIC_12,DIC_13,DIC_14,DIC_15,DIC_16,DIC_17,DIC_18,DIC_19,DIC_20,DIC_21,DIC_22,SV_sum_all_events,ECDNA_classification,dataset,HSR_classification
0,160,near-diploid,47.0,XX,7.0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,Y,MITELMAN,N
1,12754,pseudotetraploid,89.0,XXXX,6.0,NaN,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,1,0,0,1,1,1,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,Y,MITELMAN,N
2,12803,hypodiploid,44.0,XY,2.0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,Y,MITELMAN,N
3,12829,near-diploid,46.0,XX,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,

______________

## Build up dataframe of karyotype features - Karyotypes with HSR

In [12]:
df_hsr = pd.read_csv(r"C:\Users\koand\Downloads\HSR.csv")
df2 = pd.read_csv(r"C:\Users\koand\Downloads\Mitelman_dmin_data.csv")
df2h = pd.read_csv(r"C:\Users\koand\Downloads\HSR.csv")

df_hsr.head(2)

,Refno,CaseNo,InvNo,Abbreviation,Journal,Morph,Topo,KaryShort,KaryLong
0,12632,1,1,Bae et al 2008,Leuk Lymphoma,Acute myelomonocytic leukemia (FAB type M4),NaN,"46,XY,t(4;9)(p16;q22),+6,inv(9)(p11q13)c,-11,+...",NaN
1,12884,1,1,Maitta et al 2009,Cancer Genet Cytogenet,Acute myelomonocytic leukemia (FAB type M4),NaN,"45-48,XX,add(3)(p13),del(5)(q22q34),+8,add(10)...",NaN


In [ ]:
ploidy_classification = []

modal_chr_number = []

XY_chr = []

modal_chr_range = []

marker_chr_num = []

karyotype_modal_dict = {'hexaploid/octaploid':161,'hyperpentaploid':115,'pseudotetraploid':92,'triploid':69,'hypertriploid':75,'hypertetraploid':103,'near-diploid':46,'near-pseudodiploid':46,'polyploid':-1,'hypotetraploid':85,'tetraploid':92,'diploid':46,'hyperdiploid':51, 'hypodiploid':45,'pseudodiploid':46,'near-tetraploid':92,'near-triploid':69,'hypotriploid':66,'iploid':46,'aneuploid':-1, 'hyperpentaploid':121, 'hypopentaploid':110, 'pentaploid':115, 'hypohexaploid':132, 'hexaploid':138, 'heptaploid':161, 'mulitploid':-1, 'heteroploid':-1, 'decaploid':230, 'multiploidy':-1}
def categorize(value):
    if np.isnan(value):
        return np.nan
    closest_category = min(karyotype_modal_dict.keys(), key=lambda k: abs(karyotype_modal_dict[k] - value))
    return closest_category

num = ['1','2','3','4','5','6','7','8','9','10','11','12','13','14','15','16','17','18','19','20','21','22','X','Y']
chrom_loss_dict = dict()
chrom_gain_dict = dict()
for i in num:
    chrom_loss_dict[i]=[]
    chrom_gain_dict[i]=[]


cols = ['INS', 'DEL', 'ADD', 'DUP', 'TRANS', 'INV', 'DER', 'ISO', 'DIC']

chrom_dict = dict()
for i in cols:
    chrom_dict[i]=dict()
    for j in num:
        chrom_dict[i][j]=[]

ins_p = []
add_p = []
del_p=[]
dup_p =[]
inv_p=[]
iso_p=[]
dic_p=[]
trans_p=[]
der_p =[]

with_modal_range = []

for i in df_hsr.KaryShort:
    tmp = i.split(',')

    # modal chromosome number
    if '-' in tmp[0]:
        modal_chr_number.append(int(np.mean([int(o) for o in tmp[0].split('-')])))
        with_modal_range.append(tmp[0])
        
        r = [int(o) for o in tmp[0].split('-')]
        modal_chr_range.append(int(np.abs(np.subtract(r[1],r[0]))))
        
        
    else:
        modal_chr_range.append(np.nan)
        if '?' in tmp[0]:
            modal_chr_number.append(np.nan)
        else:
            modal_chr_number.append(int(tmp[0]))
            
    #XY
    if '?' in tmp[1]:
        XY_chr.append(np.nan)
    elif (tmp[1].startswith('X')) or (tmp[1].startswith('Y')):
        XY_chr.append(tmp[1])
    else:
        XY_chr.append(np.nan)
        
    #mar
    mar_counts = [k for k in tmp if ('mar' in k) and (k.startswith('+'))]
    tmp_mar = []
    
    if len(mar_counts)==0:
        marker_chr_num.append(np.nan)
        
    else:
    
        for j in set(mar_counts):
            if j==np.nan:
                tmp_mar.append(np.nan)
            else:
                ll = j.split(',')
                for k in ll:
                    if '+mar' in k:
                        tmp_mar.append(1)
                    else:
                        y = k.replace('+','').split('mar')[0]
                        if '-' in y:
                            tmp_mar.append(int(np.mean([int(o) for o in y.split('-')])))
                        else:
                            tmp_mar.append(int(y))
                            

        if (len(tmp_mar)==1) and (tmp_mar[0]==np.nan):
            marker_chr_num.append(np.nan)
        elif (len(tmp_mar)==1) and (tmp_mar[0]!=np.nan): 
            marker_chr_num.append(tmp_mar[0])
        else:
            marker_chr_num.append(np.max([k for k in tmp_mar if k!=np.nan]))
            
            
            
    #chr loss / gains
    chr_counts = [k for k in tmp if ((k.startswith('-')) or (k.startswith('+'))) and ('mar' not in k) and ('add' not in k) and ('der' not in k) and ('del' not in k) and ('hsr' not in k) and ('+i' not in k) and ('dic' not in k) and ('+t' not in k) and ('+r' not in k) and ('dup' not in k) and ('inv' not in k) and ('r' not in k) and ('?' not in k) and ('dmin' not in k) and ('ma' not in k)]
    
    tmp_gains = []
    tmp_losses = []
    
    for j in chr_counts:
        if j.startswith('-'):
            j = j.split('-')[1].split('/')[0]
            if j != '':
                tmp_losses.append(j)
        elif j.startswith('+'):
            j = j.split('+')[1].split('/')[0]
            if (j != '') and ('-' not in j) and ('de' not in j):
                tmp_gains.append(j)
        else:
            pass
            
    if len(tmp_losses)==0:
        for q in num:
            chrom_loss_dict[q].append(0)
            
    else:
        for l in set(tmp_losses):
            chrom_loss_dict[l].append(1)
            
        for n in [q for q in num if q not in tmp_losses]:
            chrom_loss_dict[n].append(0)
            
    
    if len(tmp_gains)==0:
        for q in num:
            chrom_gain_dict[q].append(0)
            
    else:
        for l in set(tmp_gains):
            chrom_gain_dict[l].append(1)
            
        for n in [q for q in num if q not in tmp_gains]:
            chrom_gain_dict[n].append(0)   
            
    
     # der chrom
    der_counts = [k for k in tmp if ((k.startswith('der')) or (k.startswith('+der'))) and ('mar' not in k) and ('+i' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    
    tmp_der = []
    if (len(der_counts) == 0 ):
        der_p.append(np.nan)
        tmp_der=[]
    else:
        der_p.append(der_counts)
        
        for j in der_counts:
            j = j.replace('?','').split('der(')[1].split(')')[0]
            
            if ';' in j:
                for k in j.split(';'):
                    if k != '':
                        tmp_der.append(k)
            else:
                if j != '':
                    tmp_der.append(j)
             
    if len(tmp_der)==0:
        for q in num:
            chrom_dict['DER'][q].append(0)
            
    elif len(tmp_der)==1:
        chrom_dict['DER'][tmp_der[0]].append(1)

        for n in [q for q in num if q != tmp_der[0]]:
            chrom_dict['DER'][n].append(0)
            
    else:
        for l in set(tmp_der):
            chrom_dict['DER'][l].append(1)
            
        for n in [q for q in num if q not in tmp_der]:
            chrom_dict['DER'][n].append(0)        
            
            
     # dup chrom

    d_counts = [k for k in tmp if ((k.startswith('dup')) or (k.startswith('+dup'))) and ('mar' not in k) and ('+i' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    tmp_counts = []
    if (len(d_counts) == 0 ):
        dup_p.append(np.nan)
        tmp_counts=[]
    else:
        dup_p.append(d_counts)
        
        
        for j in d_counts:
            j = j.replace('?','').split('dup(')[1].split(')')[0]
            
            if ';' in j:
                for k in j.split(';'):
                    if k != '':
                        tmp_counts.append(k)
            else:
                if j != '':
                    tmp_counts.append(j)
    
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['DUP'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['DUP'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['DUP'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['DUP'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['DUP'][n].append(0)                 
            
            
     # del chrom

    d_counts = [k for k in tmp if ((k.startswith('del')) or (k.startswith('+del'))) and ('mar' not in k) and ('+i' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    tmp_counts = []
    if (len(d_counts) == 0 ):
        del_p.append(np.nan)
        tmp_counts=[]
    else:
        del_p.append(d_counts)
        
        
        for j in d_counts:
            j = j.replace('?','').split('del(')[1].split(')')[0]
            
            if ';' in j:
                for k in j.split(';'):
                    if k != '':
                        tmp_counts.append(k)
            else:
                if j != '':
                    tmp_counts.append(j)
    
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['DEL'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['DEL'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['DEL'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['DEL'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['DEL'][n].append(0)                 

     # dic chrom

    d_counts = [k for k in tmp if ((k.startswith('dic')) or (k.startswith('+dic')) or (k.startswith('idic')) or (k.startswith('+idic'))) and ('mar' not in k) and ('+i' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    tmp_counts = []
    if (len(d_counts) == 0 ):
        dic_p.append(np.nan)
        tmp_counts=[]
    else:
        dic_p.append(d_counts)
        
        
        for j in d_counts:
            j = j.replace('?','').split('dic(')[1].split(')')[0]
            
            if ';' in j:
                for k in j.split(';'):
                    if k != '':
                        tmp_counts.append(k)
            else:
                if j != '':
                    tmp_counts.append(j)
    
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['DIC'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['DIC'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['DIC'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['DIC'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['DIC'][n].append(0)           
            
            
            
     # ins chrom

    d_counts = [k for k in tmp if ((k.startswith('ins')) or (k.startswith('+ins'))) and ('mar' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    tmp_counts = []
    if (len(d_counts) == 0 ):
        ins_p.append(np.nan)
        tmp_counts=[]
    else:
        ins_p.append(d_counts)
        
        
        for j in d_counts:
            j = j.replace('?','').split('ins(')[1].split(')')[0]
            
            if ';' in j:
                for k in j.split(';'):
                    if k != '':
                        tmp_counts.append(k)
            else:
                if j != '':
                    tmp_counts.append(j)
     
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['INS'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['INS'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['INS'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['INS'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['INS'][n].append(0)                       
            
            
            
     # inv chrom

    d_counts = [k for k in tmp if ((k.startswith('inv')) or (k.startswith('+inv'))) and ('mar' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    tmp_counts = []
    if (len(d_counts) == 0 ):
        inv_p.append(np.nan)
        tmp_counts=[]
    else:
        inv_p.append(d_counts)
        
        
        for j in d_counts:
            j = j.replace('?','').split('inv(')[1].split(')')[0]
            
            if ';' in j:
                for k in j.split(';'):
                    if k != '':
                        tmp_counts.append(k)
            else:
                if j != '':
                    tmp_counts.append(j)
             
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['INV'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['INV'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['INV'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['INV'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['INV'][n].append(0)                       
            
            
     # iso chrom

    d_counts = [k for k in tmp if ((k.startswith('iso')) or (k.startswith('+iso')) or (k.startswith('+i'))) and ('dic' not in k) and ('mar' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    tmp_counts = []
    if (len(d_counts) == 0 ):
        iso_p.append(np.nan)
        tmp_counts=[]
    else:
        iso_p.append(d_counts)
        
        
        for j in d_counts:
            if 'iso' in j:
                j = j.replace('?','').split('iso(')[1].split(')')[0]
            
                if ';' in j:
                    for k in j.split(';'):
                        if k != '':
                            tmp_counts.append(k)
                else:
                    if j != '':
                        tmp_counts.append(j)
                        
                        
            elif 'i(' in j:
                j = j.replace('?','').split('i(')[1].split(')')[0]
                
                if ';' in j:
                    for k in j.split(';'):
                        if k != '':
                            tmp_counts.append(k)
                else:
                    if j != '':
                        tmp_counts.append(j)      
            else:
                print(j)
    
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['ISO'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['ISO'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['ISO'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['ISO'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['ISO'][n].append(0)           
            
            
     # add chrom

    d_counts = [k for k in tmp if ((k.startswith('add')) or (k.startswith('+add'))) and ('mar' not in k) and ('+i' not in k) and ('+t' not in k) and ('+r' not in k) and ('dmin' not in k)]
    tmp_counts = []
    if (len(d_counts) == 0 ):
        add_p.append(np.nan)
        tmp_counts=[]
    else:
        add_p.append(d_counts)
        
        
        for j in d_counts:
            if '(' not in j:
                pass
            else:
                j = j.replace('?','').split('add(')[1].split(')')[0]

                if ';' in j:
                    for k in j.split(';'):
                        if k != '':
                            tmp_counts.append(k)
                else:
                    if j != '':
                        tmp_counts.append(j)
   
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['ADD'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['ADD'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['ADD'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['ADD'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['ADD'][n].append(0)  
            
            
            
# trans chrom

    d_counts = [k for k in tmp if ((k.startswith('trans')) or (k.startswith('+trans')) or (k.startswith('t')) or (k.startswith('+t'))) and ('mar' not in k) and ('+i' not in k) and ('+r' not in k) and ('dmin' not in k)]
    
    tmp_counts = []
    if (len(d_counts) == 0 ):
        trans_p.append(np.nan)
        tmp_counts=[]
    else:
        trans_p.append(d_counts)
        
        
        for j in d_counts:
            if 't(' in j:
                j = j.replace('?','').split('t(')[1].split(')')[0]
            elif 'trans(' in j:
                j = j.replace('?','').split('trans(')[1].split(')')[0]
            elif 'tas(' in j:
                j = j.replace('?','').split('tas(')[1].split(')')[0]
            if ';' in j:
                for k in j.split(';'):
                    if k != '':
                        tmp_counts.append(k)
            else:
                if j != '':
                    tmp_counts.append(j)
             
    if len(tmp_counts)==0:
        for q in num:
            chrom_dict['TRANS'][q].append(0)
            
    elif len(tmp_counts)==1:
        chrom_dict['TRANS'][tmp_counts[0]].append(1)

        for n in [q for q in num if q != tmp_counts[0]]:
            chrom_dict['TRANS'][n].append(0)
            
    else:
        for l in set(tmp_counts):
            chrom_dict['TRANS'][l].append(1)
            
        for n in [q for q in num if q not in tmp_counts]:
            chrom_dict['TRANS'][n].append(0)                       
            

print(len(df_hsr.KaryShort),len(modal_chr_number),len(modal_chr_range),len(XY_chr),len(marker_chr_num),len(der_p),len(dup_p), len(del_p), len(dic_p), len(ins_p), len(inv_p), len(iso_p), len(add_p), len(trans_p), len(with_modal_range))            
        
for n in num:
    print(len(chrom_gain_dict[n]), len(chrom_loss_dict[n]), len(chrom_dict['DER'][n]),len(chrom_dict['DUP'][n]), len(chrom_dict['DEL'][n]), len(chrom_dict['DIC'][n]), len(chrom_dict['INS'][n]), len(chrom_dict['INV'][n]), len(chrom_dict['ISO'][n]), len(chrom_dict['ADD'][n]), len(chrom_dict['TRANS'][n]))                    
            
        
df_hsr['ploidy_classification'] = modal_chr_number
df_hsr['modal chromosome number'] = modal_chr_number
df_hsr['ploidy_classification'] = df['modal chromosome number'].apply(categorize)
df_hsr['XY_chromosomes'] = XY_chr
df_hsr['marker chromosomes (average #)']=marker_chr_num
df_hsr['modal_range_numeric']=modal_chr_range            
        
    
dd = pd.DataFrame(chrom_gain_dict.items()).set_index(0).transpose()


chr_columns = [f'chr_{i}_gains' for i in range(1, 23)]

for i, col in enumerate(chr_columns, start=1):
    df_hsr[col] = dd[str(i)].tolist()[0]


dd = pd.DataFrame(chrom_loss_dict.items()).set_index(0).transpose()


chr_columns = [f'chr_{i}_loss' for i in range(1, 23)]

for i, col in enumerate(chr_columns, start=1):
    df_hsr[col] = dd[str(i)].tolist()[0]


# puts in all structural variant data
cols = ['INS', 'DEL', 'ADD', 'DUP', 'TRANS', 'INV', 'DER', 'ISO', 'DIC']
num = ['1','2','3','4','5','6','7','8','9','10','11','12','13','14','15','16','17','18','19','20','21','22']



for i in cols:

    dd = pd.DataFrame(chrom_dict[i].items()).set_index(0).transpose()
    
    for j in num:
        
        col_name = i+'_'+j

        df_hsr[col_name]= dd[j].tolist()[0]
        
# adds SV total to df

SV_all = []

tmpdf = df_hsr[df_hsr.columns.tolist()[58:]]

for i in df_hsr.index:
    
    SV_all.append(np.sum(tmpdf[tmpdf.index==i].transpose()).values[0])

df_hsr['SV_sum_all_events'] = SV_all

ec_dna_list = []

for i in df2h.KaryShort:
    if 'dmin' in i.lower():
        #print(i)
        #print(len(df2[df2['KaryShort']==i]))
        #print()
        
        ec_dna_list.append('Y')
    else:
        ec_dna_list.append('N')
        
df_hsr['ECDNA_classification'] = ec_dna_list

HSR_list = []

for i in df2h.KaryShort:
    if 'hsr' in i.lower():
        #print(i)
        #print(len(df2[df2['KaryShort']==i]))
        #print()
        
        HSR_list.append('Y')
    else:
        HSR_list.append('N')
        
df_hsr['HSR_classification'] = HSR_list

hsr_2_list = []

for i in ec_master.index:
    tmp1 = ec_master.loc[i,'ATCC_karyotype_text']
    tmp2 = ec_master.loc[i,'DSMZ_karyotype']
    
    if (pd.notna(tmp1)) and pd.notna(tmp2):
        if ('hsr' in tmp1.lower()) or ('hsr' in tmp2.lower()):
            hsr_2_list.append('Y')
        else:
            hsr_2_list.append('N')
    elif (pd.notna(tmp1)) and (pd.isna(tmp2)):
        if ('hsr' in tmp1.lower()):
            hsr_2_list.append('Y')
        else:
            hsr_2_list.append('N')
    elif (pd.isna(tmp1)) and (pd.notna(tmp2)):
        if ('hsr' in tmp2.lower()):
            hsr_2_list.append('Y')
        else:
            hsr_2_list.append('N')
    else:
        hsr_2_list.append(np.nan)
        
df_hsr['dataset'] = ['MITELMAN' for i in df_hsr.index]
df_hsr = df_hsr.drop(columns={'CaseNo','InvNo','Abbreviation','Journal','Morph','Topo','KaryShort','KaryLong'})
#df_hsr = df_hsr.drop(columns={'INS_X','INS_Y','DEL_X','DEL_Y','ADD_X','ADD_Y','DUP_X','DUP_Y','TRANS_X','TRANS_Y','INV_X','INV_Y','DER_X','DER_Y','ISO_X','ISO_Y','DIC_X','DIC_Y'})

df_hsr = df_hsr.rename(columns={'Refno':'Sample_ID'})


df_hsr.head(5)

In [14]:
len(df_hsr[df_hsr['ECDNA_classification']=='Y'])

26

In [15]:
len(df_hsr[df_hsr['HSR_classification']=='Y'])

292

In [16]:
len(df_hsr[(df_hsr['HSR_classification']=='Y')&(df_hsr['ECDNA_classification']=='N')])

266

In [17]:
len(df_hsr[(df_hsr['HSR_classification']=='Y')&(df_hsr['ECDNA_classification']=='Y')])

26

## Merge DataFrames

In [18]:
df_com = pd.concat([df,df_hsr])
df_com.head()

,Sample_ID,ploidy_classification,modal chromosome number,XY_chromosomes,marker chromosomes (average #),modal_range_numeric,chr_1_gains,chr_2_gains,chr_3_gains,chr_4_gains,chr_5_gains,chr_6_gains,chr_7_gains,chr_8_gains,chr_9_gains,chr_10_gains,chr_11_gains,chr_12_gains,chr_13_gains,chr_14_gains,chr_15_gains,chr_16_gains,chr_17_gains,chr_18_gains,chr_19_gains,chr_20_gains,chr_21_gains,chr_22_gains,chr_1_loss,chr_2_loss,chr_3_loss,chr_4_loss,chr_5_loss,chr_6_loss,chr_7_loss,chr_8_loss,chr_9_loss,chr_10_loss,chr_11_loss,chr_12_loss,chr_13_loss,chr_14_loss,chr_15_loss,chr_16_loss,chr_17_loss,chr_18_loss,chr_19_loss,chr_20_loss,chr_21_loss,chr_22_loss,INS_1,INS_2,INS_3,INS_4,INS_5,INS_6,INS_7,INS_8,INS_9,INS_10,INS_11,INS_12,INS_13,INS_14,INS_15,INS_16,INS_17,INS_18,INS_19,INS_20,INS_21,INS_22,DEL_1,DEL_2,DEL_3,DEL_4,DEL_5,DEL_6,DEL_7,DEL_8,DEL_9,DEL_10,DEL_11,DEL_12,DEL_13,DEL_14,DEL_15,DEL_16,DEL_17,DEL_18,DEL_19,DEL_20,DEL_21,DEL_22,ADD_1,ADD_2,ADD_3,ADD_4,ADD_5,ADD_6,ADD_7,ADD_8,ADD_9,ADD_10,ADD_11,ADD_12,ADD_13,ADD_14,ADD_15,ADD_16,ADD_17,ADD_18,ADD_19,ADD_20,ADD_21,ADD_22,DUP_1,DUP_2,DUP_3,DUP_4,DUP_5,DUP_6,DUP_7,DUP_8,DUP_9,DUP_10,DUP_11,DUP_12,DUP_13,DUP_14,DUP_15,DUP_16,DUP_17,DUP_18,DUP_19,DUP_20,DUP_21,DUP_22,TRANS_1,TRANS_2,TRANS_3,TRANS_4,TRANS_5,TRANS_6,TRANS_7,TRANS_8,TRANS_9,TRANS_10,TRANS_11,TRANS_12,TRANS_13,TRANS_14,TRANS_15,TRANS_16,TRANS_17,TRANS_18,TRANS_19,TRANS_20,TRANS_21,TRANS_22,INV_1,INV_2,INV_3,INV_4,INV_5,INV_6,INV_7,INV_8,INV_9,INV_10,INV_11,INV_12,INV_13,INV_14,INV_15,INV_16,INV_17,INV_18,INV_19,INV_20,INV_21,INV_22,DER_1,DER_2,DER_3,DER_4,DER_5,DER_6,DER_7,DER_8,DER_9,DER_10,DER_11,DER_12,DER_13,DER_14,DER_15,DER_16,DER_17,DER_18,DER_19,DER_20,DER_21,DER_22,ISO_1,ISO_2,ISO_3,ISO_4,ISO_5,ISO_6,ISO_7,ISO_8,ISO_9,ISO_10,ISO_11,ISO_12,ISO_13,ISO_14,ISO_15,ISO_16,ISO_17,ISO_18,ISO_19,ISO_20,ISO_21,ISO_22,DIC_1,DIC_2,DIC_3,DIC_4,DIC_5,DIC_6,DIC_7,DIC_8,DIC_9,DIC_10,DIC_11,DIC_12,DIC_13,DIC_14,DIC_15,DIC_16,DIC_17,DIC_18,DIC_19,DIC_20,DIC_21,DIC_22,SV_sum_all_events,ECDNA_classification,dataset,HSR_classification
0,160,near-diploid,47.0,XX,7.0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,Y,MITELMAN,N
1,12754,pseudotetraploid,89.0,XXXX,6.0,NaN,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,1,0,0,1,1,1,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,Y,MITELMAN,N
2,12803,hypodiploid,44.0,XY,2.0,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,Y,MITELMAN,N
3,12829,near-diploid,46.0,XX,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,

In [19]:
print(len(df_com))

874
